# Statystyka dla Data Analyst/Engineer — Moduł 15: Projekt końcowy

To ostatni moduł kursu. Zamiast kolejnej pojedynczej techniki, przejdziemy przez
**kompletną, end-to-end analizę** jednego pytania biznesowego — od pierwszego spojrzenia
na dane, przez sformułowanie i przetestowanie hipotezy, kontrolę zmiennych zakłócających,
zbudowanie modelu predykcyjnego, aż po finalny raport z rekomendacją. Po drodze użyjemy
niemal wszystkiego z Modułów 1-14.

## Spis treści
1. [Pytanie biznesowe i plan analizy](#sec1)
2. [Eksploracja danych](#sec2)
3. [Test głównej metryki: konwersja](#sec3)
4. [Wielkość efektu, moc testu, istotność praktyczna](#sec4)
5. [Efekt uboczny: czy kampania obniżyła satysfakcję?](#sec5)
6. [Próba lepszego targetowania: model predykcyjny](#sec6)
7. [Raport końcowy](#sec7)
8. [Ćwiczenia i wyzwanie końcowe](#sec8)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Pytanie biznesowe i plan analizy

**Sytuacja:** dział marketingu przetestował nową kampanię (grupa `B`) na losowo
wybranej połowie klientów, porównując ją z dotychczasowym podejściem (grupa `A`).

**Pytanie biznesowe:** *Czy powinniśmy wdrożyć kampanię B dla WSZYSTKICH klientów?*

Żeby na nie odpowiedzieć rzetelnie, potrzebujemy odpowiedzieć na kilka pytań pomocniczych:

1. Czy kampania B rzeczywiście zwiększa konwersję, czy zaobserwowana różnica to tylko
   szum próbkowania?
2. Czy efekt jest na tyle duży, żeby miał znaczenie biznesowe (nie tylko statystyczne)?
3. Czy kampania ma jakieś niepożądane skutki uboczne (np. spadek satysfakcji)?
4. Czy możemy zbudować model pomagający kierować kampanię do klientów, u których
   zadziała najlepiej?

Odpowiemy na każde z nich po kolei, dokładnie w tej kolejności.

<a id="sec2"></a>
## 2. Eksploracja danych

Zanim policzymy jakikolwiek test, spójrzmy na dane — dokładnie tak, jak zaczynaliśmy
Moduł 1.

In [ ]:
print(klienci.shape)
klienci.info()
klienci.describe()


In [ ]:
import matplotlib.pyplot as plt

fig, osie = plt.subplots(1, 2, figsize=(12, 4.5))

wskaznik_konwersji = klienci.groupby("grupa_kampanii")["konwersja"].mean()
osie[0].bar(wskaznik_konwersji.index, wskaznik_konwersji.values, color=["steelblue", "darkorange"])
osie[0].set_ylabel("Wskaznik konwersji")
osie[0].set_title("Konwersja wg grupy kampanii")

klienci.boxplot(column="satysfakcja", by="grupa_kampanii", ax=osie[1])
osie[1].set_title("Satysfakcja wg grupy kampanii")
plt.suptitle("")
plt.tight_layout()
plt.show()


Wykres słupkowy sugeruje wyraźnie wyższą konwersję w grupie B. Box plot satysfakcji
wygląda na dość podobny między grupami na pierwszy rzut oka — do tego wrócimy w sekcji
5.

<a id="sec3"></a>
## 3. Test głównej metryki: konwersja

Nasza główna metryka (ustalona z góry, zgodnie z sekcją 1 Modułu 14) to `konwersja`.
Testujemy różnicę proporcji testem chi-kwadrat (Moduł 9) i liczymy przedział ufności dla
różnicy (Moduł 6):

In [ ]:
from scipy import stats

tabela_krzyzowa = pd.crosstab(klienci["grupa_kampanii"], klienci["konwersja"])
chi2, wartosc_p, df, oczekiwane = stats.chi2_contingency(tabela_krzyzowa, correction=False)
print(f"chi2 = {chi2:.3f}, p = {wartosc_p:.4f}")


In [ ]:
podsumowanie = klienci.groupby("grupa_kampanii")["konwersja"].agg(["sum", "count", "mean"])
print(podsumowanie)

p_a, n_a = podsumowanie.loc["A", "mean"], podsumowanie.loc["A", "count"]
p_b, n_b = podsumowanie.loc["B", "mean"], podsumowanie.loc["B", "count"]

roznica = p_b - p_a
se_roznicy = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_dolny = roznica - 1.96 * se_roznicy
ci_gorny = roznica + 1.96 * se_roznicy

print(f"Roznica konwersji (B - A): {roznica:.4f}")
print(f"95% CI dla roznicy: ({ci_dolny:.4f}, {ci_gorny:.4f})")


Wartość p jest wyraźnie poniżej `0.05`, a `95%` przedział ufności dla różnicy NIE
zawiera zera — oba wskazują na tę samą konkluzję: **różnica konwersji między grupami
jest statystycznie istotna**. Możemy odrzucić H0 (brak różnicy).

<a id="sec4"></a>
## 4. Wielkość efektu, moc testu, istotność praktyczna

Istotność statystyczna to dopiero połowa odpowiedzi (Moduły 7, 11, 14). Sprawdźmy
wielkość efektu (iloraz szans, Moduł 13) i czy nasz test w ogóle miał realną szansę
wykryć efekt tej wielkości (moc, Moduł 14).

In [ ]:
from statsmodels.formula.api import logit
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

model_logistyczny = logit("konwersja ~ C(grupa_kampanii)", data=klienci).fit(disp=0)
iloraz_szans = np.exp(model_logistyczny.params["C(grupa_kampanii)[T.B]"])
print(f"Iloraz szans (B vs A): {iloraz_szans:.2f}")

wielkosc_efektu = proportion_effectsize(p_b, p_a)
moc = NormalIndPower().solve_power(
    effect_size=wielkosc_efektu, nobs1=n_a, alpha=0.05, ratio=n_b / n_a, alternative="two-sided"
)
print(f"Osiagnieta moc testu: {moc:.3f}")


**Podsumowanie liczbowe:** wskaźnik konwersji rośnie z ok. `12%` do ok. `22%` (blisko
dwukrotnie), różnica jest istotna statystycznie (`p < 0.01`), a nasz test miał solidną
moc (`> 0.8`) do wykrycia efektu tej wielkości. To nie jest efekt "istotny tylko dlatego,
że próba jest ogromna" (Moduł 14, sekcja 5) — to duża, praktycznie znacząca różnica.

<a id="sec5"></a>
## 5. Efekt uboczny: czy kampania obniżyła satysfakcję?

Zanim polecimy wdrożenie, warto sprawdzić, czy kampania B nie ma niepożądanych skutków
ubocznych. Zacznijmy od najprostszego, NAIWNEGO porównania satysfakcji między grupami:

In [ ]:
from statsmodels.formula.api import ols

model_naiwny = ols("satysfakcja ~ C(grupa_kampanii)", data=klienci).fit()
print(model_naiwny.summary().tables[1])


> ⚠️ **Niepokojący sygnał -- ale nie tak szybko**
>
> Ten prosty model sugeruje, że grupa B ma ISTOTNIE niższą satysfakcję (`p` poniżej `0.05`)! Gdybyśmy zatrzymali się tutaj, moglibyśmy błędnie wywnioskować, że kampania B poprawia konwersję kosztem zadowolenia klientów. Ale pamiętaj Moduł 11: korelacja (tu: różnica średnich) nie oznacza przyczynowości, jeśli w grze jest ZMIENNA UKRYTA. Sprawdźmy, czy obie grupy są naprawdę porównywalne pod względem innych cech.

In [ ]:
print(klienci.groupby("grupa_kampanii")["wydatki_miesieczne"].mean())


Mimo losowego przydziału do grup, czysty przypadek sprawił, że w naszej konkretnej
próbie grupa B ma nieco NIŻSZE średnie miesięczne wydatki niż grupa A — a wiemy z Modułu
11/12, że wydatki silnie i dodatnio korelują z satysfakcją. To może być właśnie ta
zmienna ukryta, która zniekształca naiwne porównanie. Skontrolujmy ją regresją liniową
(Moduł 12):

In [ ]:
model_kontrolowany = ols("satysfakcja ~ C(grupa_kampanii) + wydatki_miesieczne", data=klienci).fit()
print(model_kontrolowany.summary().tables[1])


Po skontrolowaniu wydatków, współczynnik przy `grupa_kampanii[T.B]` przestaje być
istotny statystycznie (`p` powyżej `0.05`). **Wniosek: nie ma wiarygodnych dowodów, że
kampania B pogarsza satysfakcję** — pozorny efekt z naiwnego porównania był artefaktem
przypadkowej nierównowagi wydatków między grupami, a nie prawdziwym skutkiem kampanii.

> 📊 **Zawsze sprawdzaj porównywalność grup, nawet przy losowym przydziale**
>
> Losowa alokacja gwarantuje, że grupy są porównywalne "w oczekiwaniu" (średnio, przy nieskończonej liczbie powtórzeń), ale w KONKRETNEJ, jednej próbie zawsze może się zdarzyć przypadkowa nierównowaga jakiejś cechy — szczególnie przy umiarkowanej wielkości próby. Dobra praktyka: porównaj kluczowe zmienne kontrolne między grupami i, jeśli różnią się zauważalnie, uwzględnij je jako kontrolne predyktory w modelu.

<a id="sec6"></a>
## 6. Próba lepszego targetowania: model predykcyjny

Skoro kampania B działa, czy możemy zidentyfikować, KTÓRZY klienci reagują na nią
najlepiej, żeby kierować ją precyzyjniej? Zbudujmy pełniejszy model regresji
logistycznej (Moduł 13) z dodatkowymi predyktorami:

In [ ]:
model_pelny = logit(
    "konwersja ~ C(grupa_kampanii) + C(region) + wydatki_miesieczne + wiek", data=klienci
).fit(disp=0)
print(model_pelny.summary())


> ⚠️ **Jeden pozornie istotny region -- czy powinniśmy mu ufać?**
>
> Zauważ, że jeden z regionów może wyjść z wartością p poniżej `0.05`, podczas gdy pozostałe -- nie. Zanim zaczniemy budować strategię targetowania wokół regionu, zadaj sobie pytania z Modułów 7 i 10: (1) czy istnieje jakikolwiek WIARYGODNY MECHANIZM, przez który sam region miałby wpływać na skuteczność losowo przydzielonej kampanii? W naszych danych region był przydzielony niezależnie od grupy kampanii, więc nie ma ku temu żadnego przyczynowego uzasadnienia. (2) sprawdzamy tu efekt CZTERECH kategorii regionu jednocześnie -- to dokładnie problem wielokrotnych porównań (Moduł 10): nawet gdy region NIE ma żadnego prawdziwego wpływu, szansa, że PRZYNAJMNIEJ JEDNA z czterech kategorii wyjdzie "istotna" czysto przez szum, jest znacznie wyższa niż 5%. Najbardziej prawdopodobne wyjaśnienie: to fałszywy alarm (błąd I rodzaju), a nie prawdziwy efekt.

In [ ]:
iloraz_szans_pelny = np.exp(model_pelny.params)
print(iloraz_szans_pelny)
print(f"\nWartosc p dla grupy kampanii w pelnym modelu: {model_pelny.pvalues['C(grupa_kampanii)[T.B]']:.4f}")


Kluczowa obserwacja: iloraz szans dla `grupa_kampanii[T.B]` pozostaje praktycznie
niezmieniony i istotny niezależnie od tego, czy kontrolujemy region, wydatki i wiek —
efekt kampanii jest **odporny** na dodanie innych predyktorów. Żaden z pozostałych
predyktorów nie daje spójnej, wiarygodnej podstawy do precyzyjniejszego targetowania w
tych danych — dobra wiadomość jest taka, że kampania działa mniej więcej JEDNAKOWO dobrze
dla wszystkich segmentów klientów, więc wdrożenie dla całej bazy jest rozsądne.

<a id="sec7"></a>
## 7. Raport końcowy

**Pytanie:** Czy wdrożyć kampanię B dla wszystkich klientów?

**Rekomendacja: TAK**, na podstawie następujących ustaleń:

1. Kampania B zwiększa wskaźnik konwersji z ok. `12%` do ok. `22%` — różnica jest
   statystycznie istotna (`test chi-kwadrat, p < 0.01`) i praktycznie znacząca (niemal
   dwukrotny wzrost, iloraz szans ok. `2`).
2. Test miał wystarczającą moc statystyczną (`> 0.8`), więc ten wynik nie jest
   przypadkowym szumem przy niedostatecznej próbie.
3. Naiwna analiza sugerowała spadek satysfakcji w grupie B, ale po skontrolowaniu
   przypadkowej nierównowagi wydatków między grupami efekt zniknął — **brak
   wiarygodnych dowodów na negatywny skutek uboczny**.
4. Nie znaleziono wiarygodnych podstaw do precyzyjniejszego targetowania (pozornie
   istotny efekt regionu to najprawdopodobniej fałszywy alarm) — efekt kampanii wydaje
   się spójny we wszystkich segmentach, co przemawia za pełnym wdrożeniem, a nie
   ograniczeniem do podgrupy klientów.

**Zastrzeżenia:** warto monitorować konwersję i satysfakcję również PO pełnym wdrożeniu
(nie tylko w okresie testu) oraz rozważyć dłuższy okres obserwacji satysfakcji, żeby
wykluczyć opóźnione efekty.

<a id="sec8"></a>
## 8. Ćwiczenia i wyzwanie końcowe

Ostatnie ćwiczenia tego kursu są bardziej otwarte niż wcześniej — to Twoja okazja, żeby
samodzielnie przeprowadzić kawałek analizy od początku do końca.

> 📝 **Ćwiczenie 1: Analiza dla miast zamiast regionów**
>
> Powtórz analizę z sekcji 2-3 (eksploracja + test chi-kwadrat), ale sprawdzając, czy wskaźnik konwersji różni się między `miasto` (5 kategorii) zamiast `grupa_kampanii`. Zinterpretuj wynik w świetle tego, jak zbudowaliśmy te dane.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
tabela_miasta = pd.crosstab(klienci["miasto"], klienci["konwersja"])
chi2_m, p_m, df_m, oczek_m = stats.chi2_contingency(tabela_miasta, correction=False)
print(f"chi2={chi2_m:.3f}, p={p_m:.4f}")
```

Konwersja w danych zalezy WYLACZNIE od grupa_kampanii, nie od miasta -- wiec spodziewaj sie braku istotnosci (p powyzej 0.05). Jesli akurat wyjdzie blisko granicy, to dobra okazja przypomniec sobie dyskusje o falszywych alarmach z sekcji 6.
</details>

> 📝 **Ćwiczenie 2: Przedział ufności dla ilorazu szans**
>
> Wykorzystując `model_logistyczny` z sekcji 4, wypisz `95%` przedział ufności dla współczynnika `grupa_kampanii[T.B]` (metoda `.conf_int()`), a następnie przekształć obie granice funkcją `np.exp`, żeby uzyskać przedział ufności dla ILORAZU SZANS.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
przedzial_logit = model_logistyczny.conf_int().loc["C(grupa_kampanii)[T.B]"]
przedzial_ilorazu_szans = np.exp(przedzial_logit)
print(f"95% CI dla ilorazu szans: ({przedzial_ilorazu_szans[0]:.2f}, {przedzial_ilorazu_szans[1]:.2f})")
```
</details>

> 📝 **Ćwiczenie 3: Twój własny mini-raport**
>
> Wybierz DOWOLNĄ inną parę zmiennych z `klienci` (np. `wiek` i `satysfakcja`, albo `region` i `wydatki_miesieczne`) i napisz w komórce markdown 3-4-zdaniowe podsumowanie: jaką hipotezę postawiłbyś, jaki test/model byś zastosował, i jaki wynik byś się spodziewał na podstawie tego, co wiesz o konstrukcji tych danych z wcześniejszych modułów.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# To cwiczenie jest otwarte -- nie ma jednego "poprawnego" rozwiazania.
# Przykladowa odpowiedz: "Hipoteza: wiek nie wplywa na wydatki_miesieczne (obie
# zmienne sa w danych generowane niezaleznie). Test: korelacja Pearsona (Modul 11)
# albo prosta regresja liniowa (Modul 12). Oczekiwany wynik: r bliskie 0, brak
# istotnosci statystycznej."
print("Zobacz komentarz w kodzie powyzej.")
```
</details>

> 🔥 **Wyzwanie końcowe: kompletny raport dla satysfakcji jako metryki**
>
> Przeprowadź analogiczną, kompletną analizę end-to-end jak w tym module, ale z `satysfakcja` jako GŁÓWNĄ metryką (zamiast konwersji) i `region` jako czynnikiem badanym (zamiast grupy kampanii). Uwzględnij: eksplorację (wykres), test istotności (ANOVA z Modułu 10), sprawdzenie porównywalności grup pod względem wydatków, model regresji kontrolujący wydatki, oraz krótki raport końcowy z rekomendacją (w tym przypadku np.: "czy warto projektować kampanie regionalnie zróżnicowane?").

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
from statsmodels.formula.api import ols
import statsmodels.api as sm

# 1. eksploracja
klienci.boxplot(column="satysfakcja", by="region")
plt.suptitle("")
plt.show()

# 2. test istotnosci (ANOVA)
grupy_regionow = [dane["satysfakcja"].values for _, dane in klienci.groupby("region")]
f_stat, p_anova = stats.f_oneway(*grupy_regionow)
print(f"ANOVA: F={f_stat:.3f}, p={p_anova:.6f}")

# 3. porownywalnosc grup pod wzgledem wydatkow
print(klienci.groupby("region")["wydatki_miesieczne"].mean())

# 4. model kontrolujacy wydatki
model_region = ols("satysfakcja ~ C(region) + wydatki_miesieczne", data=klienci).fit()
print(model_region.summary().tables[1])
```

W przeciwienstwie do przypadku grupa_kampanii/satysfakcja z sekcji 5, tutaj efekt regionu POZOSTAJE istotny nawet po kontroli wydatkow -- bo w Module 8 celowo wbudowalismy prawdziwy, systematyczny efekt regionu na satysfakcje (efekt_regionu), w odroznieniu od czysto przypadkowego zaklocenia z sekcji 5. To dobra ilustracja roznicy: kontrola zmiennych usuwa POZORNE efekty, ale nie usuwa PRAWDZIWYCH.
</details>

## Gratulacje!

To koniec kursu Statystyka dla Data Analyst/Engineer. W 15 modułach przeszliśmy od
podstaw statystyki opisowej, przez prawdopodobieństwo i próbkowanie, testowanie hipotez,
testy t/chi-kwadrat/ANOVA, korelację i regresję (liniową i logistyczną), aż po
projektowanie testów A/B i kompletną analizę end-to-end. Najważniejsza umiejętność, jaką
mamy nadzieję z tego wynieść, to nie zapamiętanie konkretnych wzorów czy funkcji
`scipy`/`statsmodels` — to **krytyczne myślenie o danych**: pytanie "czy to naprawdę
istotne, czy tylko szum?", "czy to naprawdę przyczyna, czy tylko korelacja?", i "czy mam
wystarczające dowody, żeby to rekomendować?".